# L3 · The same example with PyTorch autograd

**One idea:** `L.backward()` reproduces our hand-derived gradients, and gradients
**accumulate** in `.grad` — which is why training loops must call `zero_grad()`.

In [1]:
import torch

x = torch.tensor(3.0)
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
y = torch.tensor(10.0)

m = w*x
a = m + b
e = a - y
L = e**2
L.backward()

print("L      =", L.item())        # 9
print("w.grad =", w.grad.item())   # -18
print("b.grad =", b.grad.item())   # -6
assert L.item() == 9 and w.grad.item() == -18 and b.grad.item() == -6
print("autograd matches manual backprop")

L      = 9.0
w.grad = -18.0
b.grad = -6.0
autograd matches manual backprop


## Gradients accumulate — call `zero_grad` each step
Re-running backward on a fresh graph **adds** to `.grad` instead of replacing it.

In [2]:
w2 = torch.tensor(2.0, requires_grad=True)
for i in range(1, 4):
    L = (w2*x + b.detach() - y)**2
    L.backward()
    print(f"after backward #{i}: w2.grad = {w2.grad.item()}  (grows -> accumulation)")

w2.grad.zero_()
L = (w2*x + b.detach() - y)**2
L.backward()
print("after zero_grad + backward: w2.grad =", w2.grad.item())   # -18 again

after backward #1: w2.grad = -18.0  (grows -> accumulation)
after backward #2: w2.grad = -36.0  (grows -> accumulation)
after backward #3: w2.grad = -54.0  (grows -> accumulation)
after zero_grad + backward: w2.grad = -18.0


## Takeaway
- **Backprop = reverse-mode autodiff**: PyTorch matches our manual result ($L=9$, $\nabla_w=-18$, $\nabla_b=-6$).
- `.grad` **accumulates** across `backward()` calls — the conceptual reason the
  standard loop is `zero_grad()` → `backward()` → `step()`.